In [1]:
import pandas as pd

df = pd.read_csv('../clean_data/panel_interactions.csv')
df["period"] = pd.to_datetime(df["period"], format="%Y-%m")
for col in ["user_polarization", "user_homophilie"]:
    df[col] = df[col]*100
df['side_left'] = (df['user_period_side']=='Left').astype(int)
df['side_right'] = (df['user_period_side']=='Right').astype(int)

In [16]:
df.columns

Index(['user', 'period', 'n_obs', 'user_total_interactions',
       'user_total_political_interactions', 'user_period_side',
       'user_polarization', 'user_homophilie', 'user_volume_cross_inter',
       'side_left', 'side_right'],
      dtype='object')

In [2]:
from linearmodels.panel import PanelOLS

reg_df = df.set_index(["user", "period"])

model = PanelOLS.from_formula(
    "user_polarization ~ user_homophilie + side_left + side_right + user_total_interactions + EntityEffects + TimeEffects",
    data=reg_df
)

results = model.fit(cov_type="clustered", cluster_entity=True)

print(results.summary)

                          PanelOLS Estimation Summary                           
Dep. Variable:      user_polarization   R-squared:                        0.0802
Estimator:                   PanelOLS   R-squared (Between):              0.0959
No. Observations:               11262   R-squared (Within):               0.0801
Date:                Mon, May 11 2026   R-squared (Overall):              0.1128
Time:                        16:08:17   Log-likelihood                -2.139e+04
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      221.25
Entities:                        1031   P-value                           0.0000
Avg Obs:                       10.923   Distribution:                 F(4,10155)
Min Obs:                       1.0000                                           
Max Obs:                       59.000   F-statistic (robust):             25.476
                            

In [27]:
params = results.params
print("Mean, max and min polarizarion:", df['user_polarization'].mean(), df['user_polarization'].max(), df['user_polarization'].min())
print('--------------------------------------')
# variables explicatives continues (on exclut intercept et dummies d'effets fixes)
X_vars = ["user_homophilie", "user_volume_cross_inter", "side_left", "side_right", "user_total_interactions"]

for var in X_vars:
    if var in params.index:
        print(f"Average effect of {var}:", params[var] * df[var].mean())

Mean, max and min polarizarion: -0.13924782762929067 30.0 -25.0
--------------------------------------
Average effect of user_homophilie: -0.42777772865980496
Average effect of user_volume_cross_inter: 0.104861735416848
Average effect of side_left: 0.15601962235832295
Average effect of side_right: 0.09346116385066297
Average effect of user_total_interactions: -0.18512971053454924
